In [1]:
pip install torch transformers pandas scikit-learn matplotlib seaborn tqdm

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_recall_fscore_support, roc_auc_score
)
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import time
import gc
from tqdm import tqdm

# 設定 Mac 專用中文字型
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Heiti TC', 'sans-serif'] 
plt.rcParams['axes.unicode_minus'] = False

# ==========================================
# 1. 設定區 (Config)
# ==========================================
TRAIN_PKL = 'train_seg.pkl'
TEST_PKL = 'test_seg.pkl'
MODEL_NAME = 'hfl/chinese-roberta-wwm-ext'
MAX_LEN = 256
BATCH_SIZE = 32     # M4 Pro 建議 32
EPOCHS = 3          # 每一折跑 3 輪
LEARNING_RATE = 2e-5
K_FOLDS = 10        # 設定 10-Fold

# 檢測 M4 Pro (MPS)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 M4 Pro (MPS) 加速已啟用！")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("🚀 NVIDIA (CUDA) 加速已啟用！")
else:
    device = torch.device("cpu")
    print("⚠️ 使用 CPU 訓練 (會很慢)。")

# ==========================================
# 2. 資料準備 (Dataset)
# ==========================================
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        encoding = self.tokenizer.encode_plus(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def create_data_loader(df, tokenizer, max_len, batch_size):
    ds = NewsDataset(
        texts=df['text_clean'].to_numpy(),
        labels=df['label_idx'].to_numpy(),
        tokenizer=tokenizer,
        max_len=max_len
    )
    return DataLoader(ds, batch_size=batch_size, num_workers=0)

# ==========================================
# 3. 訓練與評估核心函式
# ==========================================
def train_epoch(model, data_loader, optimizer, scheduler, device, epoch_idx):
    model = model.train()
    losses = []
    correct_predictions = 0
    total_samples = 0
    
    # 簡化進度條，避免 10-Fold 輸出太多
    loop = tqdm(data_loader, leave=False, desc=f"   Train Epoch {epoch_idx}")
    
    for d in loop:
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        labels = d["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits
        
        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)
        total_samples += labels.size(0)
        losses.append(loss.item())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        loop.set_postfix(loss=loss.item())

    return correct_predictions.float() / total_samples, np.mean(losses)

def eval_model(model, data_loader, device):
    model = model.eval()
    predictions = []
    real_values = []
    
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            _, preds = torch.max(outputs.logits, dim=1)
            predictions.extend(preds.cpu().tolist())
            real_values.extend(labels.cpu().tolist())
            
    return accuracy_score(real_values, predictions)

# ==========================================
# 主程式
# ==========================================
if __name__ == '__main__':
    # --- 1. 資料處理 ---
    print("1. 讀取與處理資料...")
    train_df = pd.read_pickle(TRAIN_PKL)
    test_df = pd.read_pickle(TEST_PKL)
    
    # 還原 BERT 文本 (去除 Monpa 空白)
    train_df['text_clean'] = train_df['text'].fillna('').apply(lambda x: x.replace(' ', ''))
    test_df['text_clean'] = test_df['text'].fillna('').apply(lambda x: x.replace(' ', ''))
    
    le = LabelEncoder()
    train_df['label_idx'] = le.fit_transform(train_df['label'])
    test_df['label_idx'] = le.transform(test_df['label'])
    label_names = le.classes_
    num_classes = len(label_names)
    
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

    # --- 2. 執行 10-Fold Cross Validation ---
    print(f"\n2. 開始 {K_FOLDS}-Fold Cross Validation (這會花很長時間)...")
    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
    
    cv_accuracies = []
    fold_start_total = time.time()

    # 這裡只用 train_df 做 CV
    X = train_df
    y = train_df['label_idx']

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔹 Fold {fold + 1}/{K_FOLDS}")
        
        # 分割資料
        fold_train = X.iloc[train_idx]
        fold_val = X.iloc[val_idx]
        
        train_loader = create_data_loader(fold_train, tokenizer, MAX_LEN, BATCH_SIZE)
        val_loader = create_data_loader(fold_val, tokenizer, MAX_LEN, BATCH_SIZE)
        
        # 每次 Fold 都要重新初始化模型
        model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes)
        model = model.to(device)
        optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
        total_steps = len(train_loader) * EPOCHS
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
        
        # 訓練 Loop
        for epoch in range(EPOCHS):
            train_acc, train_loss = train_epoch(model, train_loader, optimizer, scheduler, device, epoch+1)
            # print(f"   Epoch {epoch+1}: Loss {train_loss:.4f} | Acc {train_acc:.4f}")
        
        # 驗證
        val_acc = eval_model(model, val_loader, device)
        cv_accuracies.append(val_acc)
        print(f"   ✅ Fold {fold+1} Val Accuracy: {val_acc:.4f}")
        
        # 清理記憶體 (重要！)
        del model, optimizer, scheduler
        if torch.backends.mps.is_available():
             pass # MPS 目前沒有明確的 empty_cache，依賴 GC
        elif torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    avg_cv_acc = np.mean(cv_accuracies)
    print(f"\n🏆 {K_FOLDS}-Fold CV 平均準確率: {avg_cv_acc:.4f}")
    print(f"   (CV 總耗時: {(time.time() - fold_start_total)/60:.1f} 分鐘)")


    # --- 3. 最終全量訓練 (Final Training) ---
    print(f"\n3. 使用完整訓練集進行最終模型訓練 (Epochs: {EPOCHS})...")
    final_start = time.time()
    
    full_train_loader = create_data_loader(train_df, tokenizer, MAX_LEN, BATCH_SIZE)
    test_loader = create_data_loader(test_df, tokenizer, MAX_LEN, BATCH_SIZE)
    
    final_model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes)
    final_model = final_model.to(device)
    optimizer = AdamW(final_model.parameters(), lr=LEARNING_RATE)
    total_steps = len(full_train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    
    for epoch in range(EPOCHS):
        print(f"   Final Epoch {epoch+1}/{EPOCHS}")
        train_acc, train_loss = train_epoch(final_model, full_train_loader, optimizer, scheduler, device, epoch+1)
        print(f"   -> Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
        
    final_train_time = time.time() - final_start

    # --- 4. 最終測試集評估與報表 ---
    print("\n4. 最終測試集評估 (詳細報表)...")
    
    # 為了產生混淆矩陣與 AUROC，這裡手寫評估邏輯
    final_model.eval()
    predictions, probs_list, real_values = [], [], []
    
    with torch.no_grad():
        for d in tqdm(test_loader, desc="Testing"):
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)

            outputs = final_model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            predictions.extend(preds.cpu().tolist())
            probs_list.extend(probs.cpu().tolist())
            real_values.extend(labels.cpu().tolist())

    y_true = np.array(real_values)
    y_pred = np.array(predictions)
    y_proba = np.array(probs_list)

    # 計算指標
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    try:
        auroc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='weighted')
    except:
        auroc = 0.0

    # 輸出總排行表 (含 CV 分數)
    print("\n" + "="*80)
    print("🏆 模型評估總結 (Model Ranking Summary)")
    print("="*80)
    summary_df = pd.DataFrame([{
        'Model': 'RoBERTa-wwm-ext (M4 Pro)',
        'CV Avg Acc': avg_cv_acc,
        'Test Acc': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-score': f1,
        'AUROC': auroc,
        'Final Train Time(s)': final_train_time
    }])
    print(summary_df.to_string(float_format="{:.4f}".format, index=False))

    # 詳細報表
    print("\n" + "="*80)
    print(f"📊 模型詳情: RoBERTa-wwm-ext")
    print("="*80)
    print(f"10-Fold CV Accuracy: {avg_cv_acc:.4f}")
    class_report = classification_report(y_true, y_pred, target_names=label_names, output_dict=True, zero_division=0)
    print(pd.DataFrame(class_report).transpose().to_string(float_format="{:.4f}".format))

    # 混淆矩陣圖
    print(f"\n[ 混淆矩陣圖 ]")
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names)
    plt.title(f'RoBERTa Confusion Matrix\nTest Acc: {acc:.4f} | CV Acc: {avg_cv_acc:.4f}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

🚀 M4 Pro (MPS) 加速已啟用！
1. 讀取與處理資料...

2. 開始 10-Fold Cross Validation (這會花很長時間)...

🔹 Fold 1/10


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/chinese-roberta-wwm-ext and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
   Train Epoch 1:   3%|▎           | 31/1064 [00:40<22:26,  1.30s/it, loss=1.54]